# Blinkit Campaign Runtime Tracker
Runs every hour. Detects when a campaign goes ON_HOLD (budget exhausted) and logs how many hours it ran and at what time the budget exhausted.

In [ ]:
import pandas as pd
import numpy as np
import psycopg2
import requests
import yagmail
import platform
import time
import json
import imaplib
import email
import traceback
import warnings
from bs4 import BeautifulSoup
from datetime import datetime, timedelta
from sqlalchemy import create_engine
from psycopg2.extras import execute_values
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

warnings.filterwarnings('ignore')
print('Libraries loaded')

In [ ]:
# File location
if platform.system() == 'Windows':
    FL = {'location': r'C:\Users\Amit Singh\Documents\Python_Scripts'}
elif platform.system() == 'Linux':
    FL = {'location': r'/home/misauto/Python_Scripts'}

print(f"Platform: {platform.system()} | Scripts path: {FL['location']}")

In [ ]:
# Load credentials
if platform.system() == 'Windows':
    with open(r"%s\Voylla_Cred.txt" % FL['location'], 'r') as f:
        lines = [item.strip() for item in f.readlines()]
        Voylla_config = {
            'host': lines[0], 'user': lines[1], 'password': lines[2],
            'database': lines[3], 'port': int(lines[4])
        }
    with open(r"%s\Automationalert_emailid_pass.txt" % FL['location'], 'r') as f:
        lines = [item.strip() for item in f.readlines()]
        Alert_email, Alert_password, Alert_sender_name = lines[0], lines[1], lines[2]
    with open(r"%s\automation_mail_receiver.txt" % FL['location'], 'r') as f:
        receiver_email = [item.strip() for item in f.readlines()]

elif platform.system() == 'Linux':
    with open(r'/home/misauto/Python_Scripts/Voylla_Cred.txt') as f:
        lines = [item.strip() for item in f.readlines()]
        Voylla_config = {
            'host': lines[0], 'user': lines[1], 'password': lines[2],
            'database': lines[3], 'port': int(lines[4])
        }
    with open(r'/home/misauto/Python_Scripts/Automationalert_emailid_pass.txt') as f:
        lines = [item.strip() for item in f.readlines()]
        Alert_email, Alert_password, Alert_sender_name = lines[0], lines[1], lines[2]
    with open(r'/home/misauto/Python_Scripts/automation_mail_receiver.txt') as f:
        receiver_email = [item.strip() for item in f.readlines()]

print('Credentials loaded')

In [ ]:
# DB connection with retry
MAX_RETRIES = 3
RETRY_DELAY = 5

def retry_call(func, retries=3, delay=5, label='', alert_on_fail=False):
    for attempt in range(1, retries + 1):
        try:
            return func()
        except Exception as e:
            print(f"[{label}] Attempt {attempt}/{retries} failed: {e}")
            if attempt < retries:
                time.sleep(delay)
            else:
                if alert_on_fail:
                    try:
                        send_alert_email(
                            f"Blinkit Runtime Tracker - {label} Failed",
                            f"All {retries} attempts failed for [{label}].\n\nError: {e}\n\n{traceback.format_exc()}"
                        )
                    except Exception as mail_err:
                        print(f"Alert email also failed: {mail_err}")
                raise

conn_voylla = retry_call(
    lambda: psycopg2.connect(**Voylla_config),
    label='DB Connection'
)
db_connection = create_engine(
    f"postgresql+psycopg2://{Voylla_config['user']}:{Voylla_config['password']}"
    f"@{Voylla_config['host']}:{Voylla_config['port']}/{Voylla_config['database']}"
)
print(f"DB connected at {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")


In [ ]:
# Set up / migrate Blinkit_Campaign_Runtime table
# Handles both: fresh create AND migrating old single-column PK to composite PK

setup_sql = """
DO $$
DECLARE
    pk_cols TEXT;
BEGIN
    -- Step 1: Create table if it doesn't exist yet
    CREATE TABLE IF NOT EXISTS voylla."Blinkit_Campaign_Runtime" (
        campaign_id       VARCHAR(50),
        brand             VARCHAR(100),
        log_date          DATE          NOT NULL DEFAULT CURRENT_DATE,
        last_status       VARCHAR(50),
        last_active_time  TIMESTAMP,
        last_checked      TIMESTAMP,
        budget            NUMERIC,
        exhaustion_time   TIMESTAMP,
        total_hours       NUMERIC(5,2),
        PRIMARY KEY (campaign_id, log_date)
    );

    -- Step 2: Add log_date column if missing (very old schema)
    IF NOT EXISTS (
        SELECT 1 FROM information_schema.columns
        WHERE table_schema = 'voylla'
          AND table_name   = 'Blinkit_Campaign_Runtime'
          AND column_name  = 'log_date'
    ) THEN
        ALTER TABLE voylla."Blinkit_Campaign_Runtime"
        ADD COLUMN log_date DATE NOT NULL DEFAULT CURRENT_DATE;
    END IF;

    -- Step 3: Fill any NULL log_dates so PK can be enforced
    UPDATE voylla."Blinkit_Campaign_Runtime"
    SET log_date = CURRENT_DATE
    WHERE log_date IS NULL;

    -- Step 4: Check current PK columns
    SELECT string_agg(a.attname, ',' ORDER BY array_position(i.indkey, a.attnum))
    INTO pk_cols
    FROM pg_index i
    JOIN pg_class c ON c.oid = i.indrelid
    JOIN pg_namespace n ON n.oid = c.relnamespace
    JOIN pg_attribute a ON a.attrelid = c.oid AND a.attnum = ANY(i.indkey)
    WHERE n.nspname = 'voylla'
      AND c.relname = 'Blinkit_Campaign_Runtime'
      AND i.indisprimary;

    -- Step 5: If PK is NOT already (campaign_id, log_date), migrate it
    IF pk_cols IS NOT NULL AND pk_cols != 'campaign_id,log_date' THEN
        -- Drop old PK
        EXECUTE (
            SELECT 'ALTER TABLE voylla."Blinkit_Campaign_Runtime" DROP CONSTRAINT ' || quote_ident(conname)
            FROM pg_constraint
            JOIN pg_class ON pg_class.oid = conrelid
            JOIN pg_namespace ON pg_namespace.oid = pg_class.relnamespace
            WHERE pg_namespace.nspname = 'voylla'
              AND pg_class.relname = 'Blinkit_Campaign_Runtime'
              AND contype = 'p'
            LIMIT 1
        );
        -- Add composite PK
        ALTER TABLE voylla."Blinkit_Campaign_Runtime"
        ADD PRIMARY KEY (campaign_id, log_date);
        RAISE NOTICE 'PK migrated to (campaign_id, log_date)';
    ELSE
        RAISE NOTICE 'PK already correct: %', pk_cols;
    END IF;
END $$;
"""

with conn_voylla.cursor() as cur:
    cur.execute(setup_sql)
    conn_voylla.commit()
print("Table Blinkit_Campaign_Runtime ready — PK: (campaign_id, log_date)")


In [ ]:
# Helper functions

def send_alert_email(subject, body):
    try:
        with yagmail.SMTP(Alert_email, Alert_password) as yag:
            yag.send(
                to=receiver_email,
                subject=subject,
                contents=body,
                headers={"From": f"{Alert_sender_name} <{Alert_email}>"}
            )
        print(f"Alert sent: {subject}")
    except Exception as e:
        print(f"Email failed: {e}")


def extract_firebase_token(driver):
    logs = driver.get_log("performance")
    for entry in logs:
        message = json.loads(entry["message"])["message"]
        if message["method"] == "Network.requestWillBeSent":
            headers = message.get("params", {}).get("request", {}).get("headers", {})
            if "firebase_user_token" in headers:
                return headers["firebase_user_token"]
    return None


def extract_sign_in_link(email_id, password, sender, to_email):
    IMAP_SERVER = 'imap.gmail.com'
    mail = imaplib.IMAP4_SSL(IMAP_SERVER)
    mail.login(email_id, password)
    mail.select('INBOX')
    since_date = (datetime.now() - timedelta(days=1)).strftime('%d-%b-%Y')
    search_criteria = (
        f'(FROM "{sender}" TO "{to_email}" '
        f'SUBJECT "Blinkit Brand Central" SINCE "{since_date}")'
    )
    status, messages = mail.search(None, search_criteria)
    if status != 'OK':
        mail.logout()
        raise Exception('Email search failed')
    email_ids = messages[0].split()
    if not email_ids:
        mail.logout()
        raise Exception('No email found')
    status, msg_data = mail.fetch(email_ids[-1], '(RFC822)')
    if status != 'OK':
        mail.logout()
        raise Exception('Failed to fetch email')
    msg = email.message_from_bytes(msg_data[0][1])
    sign_in_link = None
    for part in msg.walk():
        if part.get_content_type() == 'text/html':
            html = part.get_payload(decode=True)
            if not html:
                continue
            soup = BeautifulSoup(html.decode(errors='ignore'), 'html.parser')
            for a in soup.find_all('a', href=True):
                if 'sign in to your account' in a.get_text(strip=True).lower():
                    sign_in_link = a['href']
                    break
        if sign_in_link:
            break
    mail.logout()
    if not sign_in_link:
        raise Exception('Sign-in link not found')
    return sign_in_link


def human_typing_advanced(element, text):
    for char in text:
        element.send_keys(char)
        time.sleep(0.05 + 0.1 * (len(text) % 3 == 0))


print('Helper functions defined')

In [ ]:
# Fetch brands from DB
brand_query = """
SELECT * FROM "DataWarehouse"."Brand_user" a
WHERE a."channel" = 'Email_Pass for Blinkit'
AND brand != 'Petcrux'
"""
df_brands = pd.read_sql(brand_query, db_connection)
print(f"Brands fetched: {len(df_brands)}")
df_brands.head()

In [ ]:
# Load today's and yesterday's rows using Python dates (timezone-safe)
# CURRENT_DATE in PostgreSQL uses UTC — Python local time may already be next day
now       = datetime.now()
today     = now.date()
yesterday = today - timedelta(days=1)
today_midnight = datetime.combine(today, datetime.min.time())

try:
    df_today = pd.read_sql(
        f'SELECT * FROM voylla."Blinkit_Campaign_Runtime" WHERE log_date = \'{today}\'',
        db_connection
    )
    print(f"Today ({today}) rows: {len(df_today)}")
except Exception as e:
    df_today = pd.DataFrame(columns=[
        "campaign_id","brand","log_date","last_status","last_active_time",
        "last_checked","budget","exhaustion_time","total_hours"
    ])
    print(f"No today rows: {e}")

try:
    df_yesterday = pd.read_sql(
        f'SELECT * FROM voylla."Blinkit_Campaign_Runtime" WHERE log_date = \'{yesterday}\'',
        db_connection
    )
    print(f"Yesterday ({yesterday}) rows: {len(df_yesterday)}")
except Exception as e:
    df_yesterday = pd.DataFrame(columns=[
        "campaign_id","brand","log_date","last_status","last_active_time",
        "last_checked","budget","exhaustion_time","total_hours"
    ])
    print(f"No yesterday rows: {e}")

# Cleanup: clear bad carryover exhaustion on today's rows
# (rows written with old exhaustion_time from a previous day)
try:
    with conn_voylla.cursor() as cur:
        cur.execute("""
            UPDATE voylla."Blinkit_Campaign_Runtime"
            SET exhaustion_time = NULL, total_hours = NULL
            WHERE log_date = %s
              AND exhaustion_time IS NOT NULL
              AND exhaustion_time::date < %s
              AND campaign_id IN (
                  SELECT campaign_id FROM voylla."Blinkit_Campaign_Runtime"
                  WHERE log_date = %s
                    AND last_status IN ('ON_HOLD','BUDGET_EXHAUSTED','PAUSED_BUDGET')
              )
        """, (today, today, yesterday))
        conn_voylla.commit()
        if cur.rowcount:
            print(f"Cleanup: cleared {cur.rowcount} bad carryover rows for {today}")
            # Reload df_today after cleanup
            df_today = pd.read_sql(
                f'SELECT * FROM voylla."Blinkit_Campaign_Runtime" WHERE log_date = \'{today}\'',
                db_connection
            )
except Exception as e:
    print(f"Cleanup skipped: {e}")


In [ ]:
# Main loop: fresh status every run, one row per campaign per day
import numpy as np

status_updates   = []
on_hold_statuses = {"ON_HOLD", "BUDGET_EXHAUSTED", "PAUSED_BUDGET"}

def to_none(val):
    try:
        if pd.isnull(val): return None
    except Exception: pass
    return val

def to_dt(val):
    try:
        if pd.isnull(val): return None
    except Exception: pass
    if isinstance(val, datetime): return val
    if isinstance(val, pd.Timestamp): return val.to_pydatetime()
    if isinstance(val, np.datetime64): return pd.Timestamp(val).to_pydatetime()
    return val

for _, brand_row in df_brands.iterrows():
    brand      = brand_row["brand"]
    email_id   = brand_row["email"]
    password   = brand_row["password"]
    ad_account = brand_row["ad_account_id"]
    sender     = brand_row["Sender"]

    print(f"\n--- Processing brand: {brand} ---")

    try:
        campaign_query = (
            'SELECT DISTINCT "Campaign ID" '
            'FROM voylla."Blinkit_Ads_Report" '
            "WHERE TO_TIMESTAMP(\"Date\",'YYYY-MM-DD HH24:MI:SS') >= CURRENT_DATE - INTERVAL '7 days' "
            f'AND "Brand" = \'{brand}\' '
        )
        campaign_df  = retry_call(lambda: pd.read_sql(campaign_query, db_connection),
                                  retries=5, delay=5, label=f"campaign fetch {brand}", alert_on_fail=True)
        campaign_ids = campaign_df["Campaign ID"].tolist()
        print(f"  Campaigns: {campaign_ids}")
    except Exception as e:
        print(f"  Campaign fetch failed: {e}")
        continue

    driver = None; token = None; cf_cookies = {}; auth_success = False

    for auth_attempt in range(1, 6):
        try:
            if driver:
                try: driver.quit()
                except: pass
                driver = None

            print(f"  Auth attempt {auth_attempt}/5...")
            options = Options()
            options.add_argument("--disable-blink-features=AutomationControlled")
            options.set_capability("goog:loggingPrefs", {"performance": "ALL"})
            driver = webdriver.Chrome(options=options)
            driver.execute_cdp_cmd("Network.enable", {})
            driver.get("https://brands.blinkit.com/")
            time.sleep(3)

            wait = WebDriverWait(driver, 20)
            wait.until(EC.element_to_be_clickable((By.XPATH, "//button[.//div[text()='Login']]"))).click()
            username = wait.until(EC.visibility_of_element_located((By.CSS_SELECTOR, "input[placeholder='Email']")))
            human_typing_advanced(username, ad_account)
            wait.until(EC.element_to_be_clickable((By.XPATH, "//span[text()='Request Sign in Link']"))).click()

            print("  Waiting for sign-in email...")
            time.sleep(20)

            link = retry_call(lambda: extract_sign_in_link(email_id, password, sender, ad_account),
                              retries=5, delay=15, label=f"sign-in link [{brand}]", alert_on_fail=True)
            driver.get(link)
            time.sleep(3)
            driver.get("https://brands.blinkit.com/diy/list")
            time.sleep(3)

            token = extract_firebase_token(driver)
            if not token:
                raise Exception("Firebase token not found")

            cf_cookies = {c["name"]: c["value"] for c in driver.get_cookies()
                          if c["name"] in ["__cf_bm", "__cfruid", "_cfuvid"]}
            auth_success = True
            print(f"  Token captured on attempt {auth_attempt}")
            break

        except Exception as e:
            print(f"  Auth attempt {auth_attempt}/5 failed: {e}")
            if auth_attempt < 5:
                time.sleep(10)
            else:
                send_alert_email(
                    f"Blinkit Runtime Tracker - Auth Failed [{brand}]",
                    f"All 5 auth attempts failed for {brand} at {now}.\n\n{traceback.format_exc()}"
                )

    if not auth_success:
        if driver:
            try: driver.quit()
            except: pass
        continue

    for campaign_id in campaign_ids:
        try:
            get_url     = f"https://brands.blinkit.com/adservice/v1/campaigns/{campaign_id}"
            get_headers = {"accept": "application/json", "user-agent": "Mozilla/5.0",
                           "referer": f"https://brands.blinkit.com/diy/campaign/{campaign_id}",
                           "firebase_user_token": token}
            resp = retry_call(
                lambda: requests.get(get_url, headers=get_headers, cookies=cf_cookies, timeout=30),
                retries=5, delay=5, label=f"GET campaign {campaign_id}", alert_on_fail=True)
            resp_json = resp.json()

            camp_obj       = resp_json.get("data", {}).get("campaign", {})
            current_status = str(camp_obj.get("status") or camp_obj.get("campaign_status") or
                                 camp_obj.get("state") or "UNKNOWN").upper()
            current_budget = camp_obj.get("campaign_budget")
            print(f"  Campaign {campaign_id}: {current_status} | budget={current_budget}")

            # Lookup today's existing row and yesterday's row
            today_row  = df_today[df_today["campaign_id"] == str(campaign_id)]
            yest_row   = df_yesterday[df_yesterday["campaign_id"] == str(campaign_id)]
            has_today  = len(today_row) > 0
            had_yest   = len(yest_row) > 0

            prev_budget   = to_none(today_row["budget"].values[0]) if has_today else (
                            to_none(yest_row["budget"].values[0])  if had_yest else None)
            budget_to_log = float(current_budget) if (current_budget and current_budget != 260) \
                            else (float(prev_budget) if prev_budget is not None else None)

            last_active_time = today_midnight
            exhaustion_time  = None
            total_hours      = None

            if current_status in on_hold_statuses:
                existing_exhaustion = to_dt(to_none(today_row["exhaustion_time"].values[0])) if has_today else None
                today_prev_status   = str(today_row["last_status"].values[0]).upper() if has_today else None
                yest_status         = str(yest_row["last_status"].values[0]).upper() if had_yest else None

                # Exhaustion is valid only if it was recorded TODAY (not a carryover from yesterday)
                today_exhaustion_valid = (
                    existing_exhaustion is not None and
                    existing_exhaustion.date() == today
                )

                if today_exhaustion_valid:
                    # Already properly recorded today — preserve, no re-alert
                    exhaustion_time = existing_exhaustion
                    total_hours     = to_none(today_row["total_hours"].values[0])
                    total_hours     = float(total_hours) if total_hours is not None else None
                    print(f"  ON_HOLD (logged at {exhaustion_time.strftime('%I:%M %p')}): {campaign_id}")

                elif (today_prev_status in on_hold_statuses) or \
                     (yest_status in on_hold_statuses and not has_today):
                    # Carryover: was ON_HOLD in previous run today OR was ON_HOLD yesterday (first run)
                    exhaustion_time = None
                    total_hours     = None
                    print(f"  ON_HOLD (carryover, did not exhaust today): {campaign_id}")

                else:
                    # Campaign was ACTIVE/STOPPED today and just went ON_HOLD — real exhaustion
                    exhaustion_time = now
                    total_hours     = round((now - today_midnight).total_seconds() / 3600, 2)
                    print(f"  >>> BUDGET EXHAUSTED: {campaign_id} | {total_hours} hrs | budget {budget_to_log}")
                    send_alert_email(
                        f"Blinkit Campaign Budget Exhausted - {brand} | Campaign {campaign_id}",
                        (f"Brand         : {brand}\n"
                         f"Campaign ID   : {campaign_id}\n"
                         f"Date          : {today}\n"
                         f"Started at    : {today_midnight.strftime('%I:%M %p')}\n"
                         f"Exhausted at  : {now.strftime('%I:%M %p')}\n"
                         f"Total Runtime : {total_hours} hrs\n"
                         f"Budget        : Rs.{budget_to_log}")
                    )

            else:
                # Running — no exhaustion data
                exhaustion_time = None
                total_hours     = None

            status_updates.append({
                "campaign_id"     : str(campaign_id),
                "brand"           : brand,
                "log_date"        : today,
                "last_status"     : current_status,
                "last_active_time": last_active_time,
                "last_checked"    : now,
                "budget"          : budget_to_log,
                "exhaustion_time" : exhaustion_time,
                "total_hours"     : total_hours
            })

        except Exception as e:
            print(f"  Error on campaign {campaign_id}: {e}")
            send_alert_email(
                f"Blinkit Runtime Tracker - Error [{brand} | {campaign_id}]",
                f"Error: {e}\n\n{traceback.format_exc()}"
            )
            traceback.print_exc()

    if driver:
        try: driver.quit()
        except: pass

print(f"\nDone. Updates queued: {len(status_updates)}")


In [ ]:
# merged into upsert cell below
pass


In [ ]:
# Upsert: one fresh row per campaign per day
# exhaustion_time: always overwrite (Python preserve-logic reads df_today)
# total_hours: always overwrite — keeps 'hours so far' updated each run
if status_updates:
    upsert_sql = """
        INSERT INTO voylla."Blinkit_Campaign_Runtime"
            (campaign_id, brand, log_date, last_status, last_active_time,
             last_checked, budget, exhaustion_time, total_hours)
        VALUES %s
        ON CONFLICT (campaign_id, log_date) DO UPDATE SET
            brand            = EXCLUDED.brand,
            last_status      = EXCLUDED.last_status,
            last_active_time = EXCLUDED.last_active_time,
            last_checked     = EXCLUDED.last_checked,
            budget           = COALESCE(EXCLUDED.budget, \"Blinkit_Campaign_Runtime\".budget),
            exhaustion_time  = EXCLUDED.exhaustion_time,
            total_hours      = EXCLUDED.total_hours
    """
    rows = [
        (r["campaign_id"], r["brand"], r["log_date"], r["last_status"],
         r["last_active_time"], r["last_checked"], r["budget"],
         r["exhaustion_time"], r["total_hours"])
        for r in status_updates
    ]
    with conn_voylla.cursor() as cur:
        execute_values(cur, upsert_sql, rows)
        conn_voylla.commit()

    df_display = pd.DataFrame(status_updates)
    print(f"Upserted {len(df_display)} rows into Blinkit_Campaign_Runtime")
    print(df_display[["campaign_id","brand","log_date","last_status","last_active_time",
                       "exhaustion_time","total_hours","budget"]].to_string(index=False))


In [ ]:
# Log script run to python_log
st = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
SD = pd.DataFrame([{
    'Script_Name' : 'Blinkit_Campaign_Runtime_Tracker.ipynb',
    'Output'      : 'Table',
    'Table Name'  : '"Blinkit_Campaign_Runtime"',
    'Sheet_id'    : '-',
    'Report_Name' : '-',
    'Updated_at'  : st
}])

retry_call(
    lambda: SD.to_sql('python_log', db_connection, schema='voylla', if_exists='append', index=False),
    label='python_log'
)
print(f"python_log updated at {st}")